# 04. Stage 2: candidate Ranking (LightGBM Ranker)

Notebook này xây dựng mô hình chấm điểm chi tiết (Ranking) để chọn ra các bộ phim tốt nhất từ danh sách ứng viên thô được tạo ra từ giai đoạn Retrieval.

---

### Phân tích Quyết định Thiết kế:
*   **Tại sao chọn LightGBM LambdaRank?**
    *   LightGBM thuộc nhóm Gradient Boosting Decision Trees (GBDT) - là giải thuật chuẩn công nghiệp tốt nhất cho dữ liệu cấu trúc bảng (tabular). Biến thể **LambdaRank** tối ưu hóa trực tiếp hàm mục tiêu NDCG (thay vì tối ưu MSE/BCE đơn thuần), giúp xếp hạng các phim được yêu thích thực sự lên đầu danh sách hiệu quả hơn. Thuật toán phân tách dựa trên histogram giúp tốc độ train cực nhanh trên CPU.
*   **Tại sao không chọn Logistic Regression?**
    *   Hồi quy tuyến tính hoặc Logistic chỉ học được các mối quan hệ tuyến tính giữa các đặc trưng, trừ khi lập trình viên tự thiết kế các thuộc tính chéo (feature crosses) một cách thủ công và phức tạp. GBDT tự động phát hiện các mối quan hệ tương tác phi tuyến và giao cắt đặc trưng thông qua các nhánh quyết định của cây.
*   **Tại sao không chọn DeepFM hay NeuMF (Deep Learning) làm Ranker chính?**
    *   Chúng ta đang xây dựng kiến trúc thuần ML. Ngoài ra, Deep Learning cần tài nguyên tính toán lớn (GPU), thời gian huấn luyện lâu hơn gấp 10 lần, và rất dễ bị quá khớp (overfit) khi tập dữ liệu huấn luyện tương đối nhỏ (~2,000 ratings).


In [1]:
import os
import pandas as pd
import numpy as np
import pickle
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

# Load dữ liệu đã xử lý
train_ratings = pd.read_csv("processed_data/train_ratings.csv")
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))
users_df = pd.read_csv(os.path.join("..", "..", "data", "simulator", "sim_users.csv"))

with open("processed_data/id_mappings.pkl", "rb") as f:
    user_to_idx, movie_to_idx, idx_to_movie = pickle.load(f)
    
with open("processed_data/user_interacted_items.pkl", "rb") as f:
    user_interacted_items = pickle.load(f)

# Load retrieval models/matrices for feature engineering
with open("models/als_model.pkl", "rb") as f:
    als_model = pickle.load(f)
    
with open("models/tfidf_matrix.pkl", "rb") as f:
    tfidf_matrix = pickle.load(f)


In [2]:
# 1. Tạo tập dữ liệu huấn luyện cho Ranker
np.random.seed(42)

ranking_data = []
all_movie_ids = list(movie_to_idx.keys())

for _, row in train_ratings.iterrows():
    u = int(row['userId'])
    pos_item = int(row['movieId'])
    rating = row['rating']
    
    label = 1 if rating >= 3.5 else 0
    ranking_data.append({'userId': u, 'movieId': pos_item, 'label': label})
    
    interacted = user_interacted_items.get(u, set())
    for _ in range(4):
        neg_item = np.random.choice(all_movie_ids)
        while neg_item in interacted:
            neg_item = np.random.choice(all_movie_ids)
        ranking_data.append({'userId': u, 'movieId': neg_item, 'label': 0})

df_rank = pd.DataFrame(ranking_data)
print(f"Tổng số dòng huấn luyện ranker: {len(df_rank)}")
print("Phân phối nhãn:")
print(df_rank['label'].value_counts())


Tổng số dòng huấn luyện ranker: 10330
Phân phối nhãn:
label
0    8681
1    1649
Name: count, dtype: int64


In [3]:
# 2. Xây dựng các đặc trưng (Feature Engineering)
movies_df['genres'] = movies_df['genres'].fillna('')
movies_df = movies_df.set_index('movieId')
users_df = users_df.set_index('user_id')

features = []
current_uid = None
user_profile_sim = None

als_user_factors = als_model.user_factors
als_item_factors = als_model.item_factors

# Đảm bảo df_rank được sắp xếp theo userId để tối ưu việc cache profile similarity
df_rank_sorted_by_user = df_rank.sort_values(by='userId')

for _, row in df_rank_sorted_by_user.iterrows():
    uid = int(row['userId'])
    mid = int(row['movieId'])
    
    movie = movies_df.loc[mid]
    popularity = movie['popularity']
    vote_average = movie['vote_average']
    
    user = users_df.loc[uid]
    favorite_genres = set(user['favorite_genres'].split('|'))
    movie_genres = set(movie['genres'].split('|'))
    
    genre_overlap = len(favorite_genres.intersection(movie_genres))
    
    try:
        release_year = int(str(movie['release_date'])[:4])
    except:
        release_year = 2010
        
    # --- Tính đặc trưng Retrieval score ---
    u_idx = user_to_idx.get(uid, None)
    m_idx = movie_to_idx.get(mid, None)
    
    als_score = als_user_factors[u_idx].dot(als_item_factors[m_idx]) if (u_idx is not None and m_idx is not None) else 0.0
    
    if uid != current_uid:
        current_uid = uid
        liked_ids = train_ratings[(train_ratings['userId'] == uid) & (train_ratings['rating'] >= 3.5)]['movieId'].tolist()
        liked_idx = [movie_to_idx[lid] for lid in liked_ids if lid in movie_to_idx]
        if liked_idx:
            user_profile_sim = cosine_similarity(tfidf_matrix[liked_idx], tfidf_matrix).mean(axis=0)
        else:
            user_profile_sim = None
            
    cb_score = user_profile_sim[m_idx] if (user_profile_sim is not None and m_idx is not None) else 0.0
        
    features.append({
        'popularity': popularity,
        'vote_average': vote_average,
        'genre_overlap': genre_overlap,
        'release_year': release_year,
        'user_activity': user['activity_level'],
        'user_bias': user['user_bias'],
        'als_score': als_score,
        'cb_score': cb_score
    })

X = pd.DataFrame(features)
y = df_rank_sorted_by_user['label']

df_rank_sorted_by_user['group_key'] = df_rank_sorted_by_user['userId']
X_sorted = X
y_sorted = y
groups = df_rank_sorted_by_user.groupby('group_key', sort=False).size().values

print(f"Features head:")
display(X_sorted.head(3))


Features head:


,popularity,vote_average,genre_overlap,release_year,user_activity,user_bias,als_score,cb_score
0,2.9697,7.139,1,2024,134,-0.069132,0.988976,0.088926
1,2.8158,5.186,1,2010,134,-0.069132,0.996257,0.072341
2,2.2224,4.167,1,2004,134,-0.069132,0.000000,0.002497


In [4]:
# 3. Huấn luyện LightGCN Ranker (LambdaRank)
ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=15,
    random_state=42
)

ranker.fit(
    X_sorted, y_sorted,
    group=groups
)

# Lưu Ranker model
with open("models/lgb_ranker.pkl", "wb") as f:
    pickle.dump(ranker, f)
    
print("Huấn luyện thành công LGBMRanker!")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000386 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1370
[LightGBM] [Info] Number of data points in the train set: 10330, number of used features: 8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Huấn luyện thành công LGBMRanker!
